# Setup

In [3]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

load_dotenv()

model_name = os.getenv("MODEL_NAME")

if not model_name:
    raise ValueError("Please set the MODEL_NAME environment variable.")

llm = AzureChatOpenAI(
    azure_deployment=model_name,
    api_version="2024-12-01-preview",
    temperature=0.0,
)

# Idea generation

In [4]:
logs = []

parser = StrOutputParser()

parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

In [5]:
idea_prompt = PromptTemplate(
    template=(
        "You are a specility of creating business iteas for "
        "a given {industry} it must be connected with the latest news "
        "you need to consider oridinality and ROI."
    )
)

idea_chain = (idea_prompt | llm | parse_and_log_output_chain)

In [6]:
idea_result = idea_chain.invoke("AI")

In [7]:
idea_result["output"]

'Absolutely — I can help generate **business ideas for a specific AI**, and I’ll make sure they are:\n\n- **Connected to current/latest news or market trends**\n- **Original / differentiated**\n- **Focused on ROI**\n- **Practical and commercially viable**\n\nTo do this well, I need a bit more information:\n\n### Please send me:\n1. **Which AI are you referring to?**\n   - Example: ChatGPT, Claude, Gemini, Midjourney, Perplexity, a custom AI model, etc.\n\n2. **What industry or market do you want ideas for?**\n   - Example: healthcare, finance, education, legal, e-commerce, real estate, marketing, HR, logistics, etc.\n\n3. **Your target customer**\n   - Example: consumers, SMBs, enterprises, creators, agencies, schools, hospitals, etc.\n\n4. **Your preferred business model**\n   - Example: SaaS, subscription, marketplace, consulting, API, licensing, lead-gen, usage-based, etc.\n\n5. **Any geography or language focus?**\n   - Example: US, EU, India, MENA, global\n\n---\n\n### What I can 

In [8]:
logs

[AIMessage(content='Absolutely — I can help generate **business ideas for a specific AI**, and I’ll make sure they are:\n\n- **Connected to current/latest news or market trends**\n- **Original / differentiated**\n- **Focused on ROI**\n- **Practical and commercially viable**\n\nTo do this well, I need a bit more information:\n\n### Please send me:\n1. **Which AI are you referring to?**\n   - Example: ChatGPT, Claude, Gemini, Midjourney, Perplexity, a custom AI model, etc.\n\n2. **What industry or market do you want ideas for?**\n   - Example: healthcare, finance, education, legal, e-commerce, real estate, marketing, HR, logistics, etc.\n\n3. **Your target customer**\n   - Example: consumers, SMBs, enterprises, creators, agencies, schools, hospitals, etc.\n\n4. **Your preferred business model**\n   - Example: SaaS, subscription, marketplace, consulting, API, licensing, lead-gen, usage-based, etc.\n\n5. **Any geography or language focus?**\n   - Example: US, EU, India, MENA, global\n\n---

# Idea analysis

In [9]:
analysis_prompt = PromptTemplate(
    template=(
        "Analyze the following business idea: "
        "Idea: {idea} "
        "Identify 3 key strengths and 3 potential weaknesses of the idea."
    )
)

In [10]:
analysis_chain = (analysis_prompt | llm | parse_and_log_output_chain)

In [11]:
analysis_result = analysis_chain.invoke(idea_result["output"])

In [44]:
analysis_result["output"]

'Here’s an analysis of the idea as a **business concept**:\n\n## What the idea is\nThis is essentially a **prompted AI business-idea generator service**: users provide an industry, trend, or target customer, and the system returns monetizable AI business ideas tied to current news and ROI.\n\nIt’s less a standalone product in its current form and more a **lead-generation / content / consulting funnel** that could evolve into a productized research tool.\n\n---\n\n## 3 key strengths\n\n### 1. Clear user value and low friction\nThe offer is easy to understand:  \n“Give me a topic, and I’ll give you relevant AI business ideas.”\n\nThat makes it accessible to:\n- founders\n- consultants\n- investors\n- corporate innovation teams\n- content creators\n\nIt reduces the blank-page problem and helps users move from vague interest to concrete opportunities quickly.\n\n### 2. Strong positioning around timeliness and ROI\nThe idea is differentiated by focusing on:\n- **current trends/news**\n- **o

# Report generation

In [12]:
report_prompt = PromptTemplate(
    template=(
        "Here is a business analysis: "
        "Strengths & Weaknesses: {output} "
        "Generate a structured business report." 
    )
)

In [13]:
class AnalysisReport(BaseModel):
    """Strengths and Weaknesses about a business idea"""
    strengths: list = Field(default=[], description="Idea's strength list")
    weaknesses: list = Field(default=[], description="Idea's weaknesse list")

In [14]:
report_chain = (
    report_prompt | llm.with_structured_output(schema=AnalysisReport, method="function_calling")
)

In [15]:
report_result = report_chain.invoke(analysis_result["output"])
report_result

AnalysisReport(strengths=['Clear value proposition: turns vague AI opportunity-seeking into concrete business ideas.', 'Strong personalization framework using AI tool, industry, customer, business model, and geography.', 'Commercially oriented output focused on trends, originality, ROI, and practical viability.', 'Useful for founders, consultants, and operators who want actionable ideas rather than generic inspiration.'], weaknesses=['Risk of being perceived as a generic AI startup idea generator in a crowded market.', 'Dependence on timely market data makes accuracy and freshness difficult to sustain.', 'Limited defensibility because similar prompt flows can be replicated easily.', 'May require sharper niche focus or proprietary methodology to stand out long term.'])

# End to End chain

In [16]:
e2e_chain = ( 
    RunnablePassthrough() 
    | idea_chain
    | RunnableParallel(idea=RunnablePassthrough())
    | analysis_chain
    | report_chain
)

In [17]:
import importlib
importlib.invalidate_caches()

In [18]:
e2e_chain.get_graph().print_ascii()

            +------------------+         
            | PassthroughInput |         
            +------------------+         
                      *                  
                      *                  
                      *                  
              +-------------+            
              | Passthrough |            
              +-------------+            
                      *                  
                      *                  
                      *                  
             +----------------+          
             | PromptTemplate |          
             +----------------+          
                      *                  
                      *                  
                      *                  
            +-----------------+          
            | AzureChatOpenAI |          
            +-----------------+          
                      *                  
                      *                  
                      *           

In [20]:
e2e_result = e2e_chain.invoke("software development")

In [21]:
e2e_result.strengths

['Clear, practical value proposition for founders and product teams',
 'Broad market applicability across multiple user segments and industries',
 'Strong alignment with current demand for AI-assisted ideation and validation']

In [22]:
e2e_result.weaknesses

['Risk of being perceived as too generic in a crowded market',
 'Difficulty proving originality and ROI accurately without a robust framework',
 'Dependence on the quality and freshness of external trend/news data']